# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import joblib
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)

### machine learning (scikit-learn)
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
from statsmodels.tsa.stattools import adfuller

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:

import smartcheck.modeling_project_specific as mps

# 3. Regression modeling

## 3.1 ACF/PACF utils

| Composante | Type                        | Doit être évaluée sur…                                 | Méthode                             | Où regarder                                    |
| ---------- | --------------------------- | ------------------------------------------------------ | ----------------------------------- | ---------------------------------------------- |
| **p**      | AR                          | Série **stationnaire** (différenciée si besoin)        | **PACF**                            | Les premiers lags (1, 2, 3...)                 |
| **q**      | MA                          | Série **stationnaire** (différenciée si besoin)        | **ACF**                             | Les premiers lags (1, 2, 3...)                 |
| **P**      | AR saisonnier               | Série **différenciée saisonnièrement**                 | **PACF**                            | Les **multiples** de la saison (ex. 24, 48...) |
| **Q**      | MA saisonnier               | Série **différenciée saisonnièrement**                 | **ACF**                             | Les **multiples** de la saison (ex. 24, 48...) |
| **d**      | Différenciation simple      | déterminée par test de **stationnarité**               | (ADF, KPSS, ou inspection visuelle) | —                                              |
| **D**      | Différenciation saisonnière | idem, sur tendance **saisonnière**                     | (souvent une saison de 24 ou 12)    | —                                              |
| **s**      | Période de saison           | définie à l’avance (ex. 24 pour données horaires/jour) | —                                   | —                                              |


In [ ]:
def affichage_pacf_acf(y_input, exp_params):
    def ajouter_lignes_saisonnieres(ax, seasonal_lags):
        for lag in seasonal_lags:
            ax.axvline(x=lag, color='red', linestyle='--', alpha=0.6)

    # Saison estimée via dictionnaire externe
    s1 = exp_params["seasonal_order"][3]

    # Décomposition saisonnière
    decomp_res = seasonal_decompose(y_input, model='additive', period=s1)
    fig1 = decomp_res.plot()
    fig1.set_figwidth(12)
    fig1.set_figheight(12)
    fig1.suptitle("Décomposition saisonnière additive", fontsize=16)
    plt.show()

    # Test ADF : stationnarité de la série d'origine
    adf_result = adfuller(y_input)
    logging.info(f"p-value ADF sans différenciation : {adf_result[1]:.4f}")

    # Différenciation simple systématique (pour permettre d'avoir un ACF/PACF lisible) et nouveau test ADF
    y = y_input.diff(1).dropna()
    adf_result_d1 = adfuller(y)
    logging.info(f"p-value ADF après différenciation simple : {adf_result_d1[1]:.4f}")

    # Différenciation saisonnière (degré 1)
    y_s1 = y.diff(s1).dropna()

    max_lag = s1 * 7  # ex: pour 7 saison horaire
    seasonal_lags = [k * s1 for k in range(1, (max_lag // s1) + 1)]

    fig2, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 20))

    # PACF / ACF sur série stationnaire (détermination p, q)
    plot_pacf(y, lags=s1, ax=ax1)
    ax1.set_title('PACF sur série stationnaire (pour déterminer p)')
    
    plot_acf(y, lags=s1, ax=ax2)
    ax2.set_title('ACF sur série stationnaire (pour déterminer q)')

    # PACF / ACF sur série avec différenciation saisonnière (P, Q)
    plot_pacf(y_s1, lags=max_lag, ax=ax3)
    ajouter_lignes_saisonnieres(ax3, seasonal_lags)
    ax3.set_title(f'PACF avec différenciation saisonnière ({s1} lags) (pour déterminer P)')

    plot_acf(y_s1, lags=max_lag, ax=ax4)
    ajouter_lignes_saisonnieres(ax4, seasonal_lags)
    ax4.set_title(f'ACF avec différenciation saisonnière ({s1} lags) (pour déterminer Q)')

    plt.tight_layout()
    plt.show()

## 3.2 Performance analysis and visualisation

In [ ]:
# On construit une liste en rechargeant les enregisrements de résultats depuis le disque
experiment_12 = "sarimax_results_exo_1-1-1_1-1-1-24_Magenta-SE-NO_-3000-end.joblib"
experiment_13 = "sarimax_results_exo_3-1-3_1-1-1-24_Magenta-SE-NO_-3000-end.joblib"
experiment_14 = "sarimax_results_exo_3-1-3_3-1-3-24_Magenta-SE-NO_-3000-end.joblib"
experiment_15 = "sarimax_results_exo_3-1-3_3-1-3-24_Magenta-SE-NO_-3500-end.joblib"
# experiment_21 = "sarimax_results_exo_3-1-3_3-1-3-24_Sebastopol-S-N_-1500-end.joblib"
# experiment_22 = "sarimax_results_exo_3-1-3_1-1-1-24_Sebastopol-S-N_1500-5000.joblib"

# il faut restituer les clés (nom_du_site_de_comptage, orientation_compteur), seule information non sauvegardée
models_sarimax_results = {
    # f"{experiment_11}":joblib.load(experiment_11),
    f"{experiment_12}":joblib.load(experiment_12),
    f"{experiment_13}":joblib.load(experiment_13),
    f"{experiment_14}":joblib.load(experiment_14),
    f"{experiment_15}":joblib.load(experiment_15),

    # f"{experiment_21}":joblib.load(experiment_21),
    # f"{experiment_22}":joblib.load(experiment_22),
}

In [ ]:
# Afficher une modélisation de compteur spécifique
models_results = models_sarimax_results
for experiment, model_results in models_results.items():
    exp_params = model_results["exp_params"]
    compteur_key = exp_params["key"]
    logging.info(f"\n{"*"*80}\nParamètres {exp_params} :\n{"*"*80}")
    dates = model_results["X_test_dates"]["date_et_heure_de_comptage_local"]  # type: ignore
    X_test_dates = model_results["X_test_dates"]  # type: ignore
    y_test = model_results["y_test"]  # type: ignore
    y_train = model_results["y_train"]  # type: ignore
    y_train_pred = model_results["y_train_pred"]  # type: ignore
    y_test_pred = model_results["y_test_pred"]  # type: ignore
    periode_limite = (
        dates[0],
        # dates[len(dates)-310],
        dates[len(dates)-1]
    )

    # Affichage des metrique train et test
    model_train_metrics = mps.compute_metrics(
        y_train,
        y_train_pred
    )
    model_test_metrics = mps.compute_metrics(
        y_test,
        y_test_pred
    )
    logging.info(f"Metriques du modèle (Train): {model_train_metrics}")
    logging.info(f"Metriques du modèle (Test):: {model_test_metrics}")

    # projection des predictions de test dans le temps
    fig_pred = mps.plot_predictions(
        str(compteur_key),
        X_test_dates, 
        y_test, 
        y_test_pred, 
        periode_limite=periode_limite
    )
    plt.show()

    # projection des résidus et calcul du coefficient de dérive dans le temps
    fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
        str(compteur_key),
        X_test_dates, 
        y_test, 
        y_test_pred.values, 
        periode_limite=periode_limite
    )
    logging.info(f"Pente de la droite de régression des résidus dans le temps (dérive) : {model_res_coeff}")
    plt.show()

    # Affichage PACF/ACF
    affichage_pacf_acf(y_train, exp_params)